# TP 3 – Image Classification with Transfer Learning
## Dataset: Food (3 classes) | Models: MobileNetV2 / VGG16 / ResNet50 / EfficientNetB0 / InceptionV3

In the previous practicals, we built models from scratch:
- **ANN**: flattened pixels → Dense layers
- **CNN**: Conv2D + MaxPooling → Dense layers

Both were trained entirely on our small Food dataset. The problem? With limited data, models struggle to learn rich, generalizable features.

**Transfer Learning** solves this by reusing a model that was already trained on a massive dataset (ImageNet — 1.2 million images, 1000 classes). This model has already learned general visual features: edges, textures, shapes, object parts. We can **transfer** this knowledge to our task.

---

### Two strategies — and we'll try both:

| Strategy | What we do | When to use |
|----------|-----------|-------------|
| **Method A – Feature Extraction** | Freeze all pretrained weights. Only train the new classifier head. | Small dataset, fast training |
| **Method B – Fine-Tuning** | Freeze early layers. Unfreeze the last few layers and train them with a very small learning rate. | More data, want higher accuracy |

### Some base models (switch by commenting/uncommenting in Steps 1 and 4):

The pretrained models listed below are built on different architectural principles. These structural differences influence computational cost, feature extraction capability, and generalization performance. Comparing them allows us to analyze the impact of architecture on transfer learning effectiveness.

| Model | Size | Speed | Accuracy | Min Input |
|-------|------|-------|----------|----------|
| **MobileNetV2** | Light | ⚡⚡⚡ | Good | 32×32 |
| **EfficientNetB0** | Light | ⚡⚡⚡ | Very good | 32×32 |
| **ResNet50** | Medium | ⚡⚡ | Very good | 32×32 |
| **InceptionV3** | Medium | ⚡⚡ | Very good | 75×75 |
| **VGG16** | Heavy | ⚡ | Good | 32×32 |

**for this work, we use the:** MobileNetV2 — lightweight and fast, ideal for experimenting.

**Dataset structure (same as before):**
```
FoodDataSet/
    train/        → Bread / Dairy product / Egg
    validation/   → Bread / Dairy product / Egg
    test/         → Bread / Dairy product / Egg
```

---
## Step 1 — Import Libraries

We import the same tools as before, plus CNN-specific Keras layers and **all five pretrained models**.
Each model has its own `preprocess_input` function — it is critical to use the one that matches your chosen model, since each was trained with a different normalization.

| Model | preprocess_input range | Notes |
|-------|----------------------|-------|
| MobileNetV2 | [-1, 1] | Default in this notebook |
| EfficientNetB0 | [0, 255] (no scaling) | Built-in normalization |
| ResNet50 | BGR, mean-subtracted | Classic preprocessing |
| InceptionV3 | [-1, 1] | Requires input ≥ 75×75 |
| VGG16 | BGR, mean-subtracted | Same as ResNet50 |

**To switch model:** uncomment the desired model import below AND update Step 4A/4B accordingly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# ─────────────────────────────────────────────────────────────────
# CHOOSE YOUR BASE MODEL — uncomment ONE model + its preprocess_input
# Keep all others commented out.
# ─────────────────────────────────────────────────────────────────

# ✅ OPTION 1: MobileNetV2 — lightweight and fast (DEFAULT)
from tensorflow.keras.applications import MobileNetV2 as BaseModel
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
MODEL_NAME = "MobileNetV2"

# ── OPTION 2: EfficientNetB0 — light but more accurate than MobileNetV2
# from tensorflow.keras.applications import EfficientNetB0 as BaseModel
# from tensorflow.keras.applications.efficientnet import preprocess_input
# MODEL_NAME = "EfficientNetB0"

# ── OPTION 3: ResNet50 — deeper, very popular, solid accuracy
# from tensorflow.keras.applications import ResNet50 as BaseModel
# from tensorflow.keras.applications.resnet50 import preprocess_input
# MODEL_NAME = "ResNet50"

# ── OPTION 4: InceptionV3 — good for complex patterns (needs IMG_SIZE >= 75)
# from tensorflow.keras.applications import InceptionV3 as BaseModel
# from tensorflow.keras.applications.inception_v3 import preprocess_input
# MODEL_NAME = "InceptionV3"

# ── OPTION 5: VGG16 — classic and simple architecture, heavier to train
# from tensorflow.keras.applications import VGG16 as BaseModel
# from tensorflow.keras.applications.vgg16 import preprocess_input
# MODEL_NAME = "VGG16"

# ─────────────────────────────────────────────────────────────────

print("Libraries imported successfully.")
print(f"Selected base model: {MODEL_NAME}")
print("TensorFlow version:", tf.__version__)

---
## Step 2 — Load the Dataset

Same structure as the ANN and CNN notebooks, with **one important difference**:

Instead of `rescale=1./255`, we use **`preprocess_input`** imported in Step 1. Each model has its own normalization that must match what was used during its ImageNet training. Using the wrong preprocessing would feed the model inputs it has never seen before, degrading performance.

We set `IMG_SIZE = (96, 96)`. This works for all models **except InceptionV3**, which requires at least **75×75**. If you switch to InceptionV3, change `IMG_SIZE` to `(96, 96)` or larger — it already works here. For VGG16 and ResNet50, a larger size like `(224, 224)` is recommended for best results but will be slower.

In [ ]:
IMG_SIZE = (96, 96)   # larger than before — more detail for the pretrained model
BATCH_SIZE = 32

# Use preprocess_input instead of rescale=1./255
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
val_datagen   = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    r"C:\Users\lenovo\Desktop\FoodDataSet\train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    r"C:\Users\lenovo\Desktop\FoodDataSet\validation",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    r"C:\Users\lenovo\Desktop\FoodDataSet\test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_generator.num_classes
print("\nClass mapping:", train_generator.class_indices)
print("Number of classes:", num_classes)

---
## Step 3 — Visualize Sample Images

Same as before — always inspect the data first. Note that after `preprocess_input`, pixel values are in [-1, 1], so we clip them back to [0, 1] for display purposes only (using `np.clip`).

In [ ]:
images, labels = next(train_generator)
class_names = list(train_generator.class_indices.keys())

plt.figure(figsize=(12, 4))
for i in range(8):
    plt.subplot(2, 4, i+1)
    # Clip to [0,1] for display since preprocess_input shifts values to [-1, 1]
    plt.imshow(np.clip(images[i] / 2 + 0.5, 0, 1))
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis('off')
plt.suptitle('Sample training images', fontsize=13)
plt.tight_layout()
plt.show()

---
---
# METHOD A — Feature Extraction (Frozen Weights)

**Idea:** Use MobileNetV2 as a fixed feature extractor. All its weights are **frozen** — they will not be updated during training. We only train the new classifier layers we add on top.

**Analogy:** Imagine MobileNetV2 is an expert photographer who already knows how to identify shapes, textures, and objects. We simply ask them to describe each food image, then we train a small classifier on those descriptions.

**When to use:** When you have a small dataset (like ours), or when you want fast training.

---
## Step 4A — Load Pretrained Base Model and Freeze It

We instantiate the model selected in Step 1 (`BaseModel`) with:
- **`weights='imagenet'`**: use weights already learned on ImageNet (1.2M images)
- **`include_top=False`**: remove the original classifier head (designed for 1000 ImageNet classes). We will add our own head for 3 food classes.
- **`input_shape=(96, 96, 3)`**: tell the model what image size to expect

Then we set **`base_model.trainable = False`** — this freezes all layers. Their weights will stay exactly as they were after ImageNet training. Only our new layers will be updated.

> **Note:** If you switched to InceptionV3, VGG16 or ResNet50, the number of total layers and parameters will be different — that's expected. The rest of the code works identically.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Load the base model selected in Step 1
# BaseModel is an alias for whichever model you uncommented there.
# No need to change anything here — just change Step 1.
# ─────────────────────────────────────────────────────────────────

base_model_A = BaseModel(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

# Freeze ALL layers of the base model — none will be updated during training
base_model_A.trainable = False

# Verify: count trainable vs non-trainable parameters
print(f"Base model: {MODEL_NAME}")
print(f"Total layers: {len(base_model_A.layers)}")
trainable_params     = sum(tf.size(w).numpy() for w in base_model_A.trainable_weights)
non_trainable_params = sum(tf.size(w).numpy() for w in base_model_A.non_trainable_weights)
print(f"Trainable parameters:     {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")

---
## Step 5A — Build the Full Model (Feature Extraction)

We stack our custom classifier head on top of the frozen base:

- **`base_model_A`**: the frozen MobileNetV2 — extracts feature maps from images
- **`GlobalAveragePooling2D()`**: averages each feature map to a single number. If the feature maps are (3×3×1280), this produces a vector of 1280 values. Much more compact than `Flatten` which would give 3×3×1280 = 11,520 values.
- **`Dense(128, activation='relu')`**: a small fully connected layer to combine the extracted features
- **`Dense(num_classes, activation='softmax')`**: output layer — one probability per food class

The key point: **only the last 3 layers will be trained**. The base model contributes features but its weights stay frozen.

In [ ]:
model_A = Sequential([
    base_model_A,                        # frozen MobileNetV2 feature extractor
    GlobalAveragePooling2D(),            # compact pooling — replaces Flatten
    Dense(128, activation="relu"),       # custom classification layer
    Dense(num_classes, activation="softmax")  # output: 3 food classes
])

model_A.summary()

---
## Step 6A — Compile and Train (Feature Extraction)

We compile exactly as before. Because only our 3 layers are trainable, training is **very fast** — we are not updating 2.2M MobileNetV2 parameters, only the ~165K parameters of our head.

We train for **10 epochs**. You'll likely see accuracy jump quickly in the first few epochs — this is because the frozen base already provides excellent features from ImageNet, and our head only needs a few passes to learn how to classify food from them.

In [ ]:
model_A.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Training Method A — Feature Extraction (frozen base)...\n")
history_A = model_A.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

---
## Step 7A — Plot Training Curves (Feature Extraction)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_A.history["accuracy"],     label="Train")
plt.plot(history_A.history["val_accuracy"], label="Validation", linestyle="--")
plt.title("Method A — Accuracy (Feature Extraction)")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_A.history["loss"],     label="Train")
plt.plot(history_A.history["val_loss"], label="Validation", linestyle="--")
plt.title("Method A — Loss (Feature Extraction)")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 8A — Evaluate and Confusion Matrix (Feature Extraction)

In [ ]:
test_loss_A, test_acc_A = model_A.evaluate(test_generator)
print(f"[Method A] Test Loss:     {test_loss_A:.4f}")
print(f"[Method A] Test Accuracy: {test_acc_A:.4f} ({test_acc_A*100:.1f}%)")

# Reset generator before prediction
test_generator.reset()
preds_A = model_A.predict(test_generator)
y_pred_A = np.argmax(preds_A, axis=1)
y_true   = test_generator.classes
labels   = list(test_generator.class_indices.keys())

cm_A = confusion_matrix(y_true, y_pred_A)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm_A, display_labels=labels).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix — Method A (Feature Extraction)")
plt.tight_layout()
plt.show()

print("\nClassification Report — Method A:")
print(classification_report(y_true, y_pred_A, target_names=labels))

---
---
# 🔧 METHOD B — Fine-Tuning (Partially Unfrozen Weights)

**Idea:** Start from the same pretrained MobileNetV2, but this time **unfreeze the last few layers** of the base model and train them alongside our classifier head, using a **very small learning rate**.

**Why unfreeze only the last layers?** MobileNetV2 learns features hierarchically:
- **Early layers** (frozen): detect generic patterns — edges, corners, colors. These are universal and should not change.
- **Late layers** (unfrozen): detect high-level, task-specific features — shapes of bread, textures of cheese. These can be adapted to our food dataset.

**Why a very small learning rate?** The pretrained weights are already good. A large learning rate would destroy them. We want to make small, careful adjustments — not random re-initialization.

**When to use:** When you want the best possible accuracy and have enough data to justify adapting the deep layers.

---
## Step 4B — Load Base Model and Selectively Unfreeze Last Layers

We load the same model again (fresh copy), then:
1. Freeze the entire base model first (`trainable = False`)
2. Then unfreeze only the **last 5 layers** (`layer.trainable = True` for layers from index -30 onward)

The last layers are the most task-specific and most worth adapting to food images. Early layers detect universal patterns (edges, corners) that should stay frozen.

> **Tip for other models:** The number of layers varies — VGG16 has 19, ResNet50 has 175, InceptionV3 has 311. Adjust `UNFREEZE_FROM` accordingly:
> - VGG16: try `-4` or `-6` (unfreeze last 1–2 blocks)
> - ResNet50: try `-30` to `-50`
> - InceptionV3: try `-30` to `-60`
> - EfficientNetB0: try `-20` to `-40`

**Try:** change `UNFREEZE_FROM` to `-20` (fewer layers) or `-50` (more layers) and compare.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Load a fresh copy of the base model selected in Step 1
# ─────────────────────────────────────────────────────────────────

base_model_B = BaseModel(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

# Step 1: freeze everything first
base_model_B.trainable = False

# Step 2: unfreeze the last N layers
# ─────────────────────────────────────────────────────────────────
# Adjust UNFREEZE_FROM depending on the model:
#   MobileNetV2  (154 layers) → try -30 (default)
#   EfficientNetB0(237 layers) → try -30 to -40
#   ResNet50     (175 layers) → try -30 to -50
#   InceptionV3  (311 layers) → try -30 to -60
#   VGG16        ( 19 layers) → try -4  to -6
# ─────────────────────────────────────────────────────────────────
UNFREEZE_FROM = -30  # try: -20, -50

for layer in base_model_B.layers[UNFREEZE_FROM:]:
    layer.trainable = True

trainable_count     = sum(1 for l in base_model_B.layers if l.trainable)
non_trainable_count = sum(1 for l in base_model_B.layers if not l.trainable)

print(f"Base model: {MODEL_NAME}")
print(f"Total layers: {len(base_model_B.layers)}")
print(f"Trainable layers (unfrozen): {trainable_count}")
print(f"Frozen layers:               {non_trainable_count}")

---
## Step 5B — Build the Full Model (Fine-Tuning)

The architecture is **identical to Method A** — same GlobalAveragePooling2D, same Dense and Dropout layers. The only difference is that now some layers of the base model are also trainable.

Notice in `model_B.summary()` that the number of **trainable parameters** will be much larger than in Method A, because we are now also updating the last 30 layers of MobileNetV2.

In [ ]:
model_B = Sequential([
    base_model_B,                         # partially unfrozen MobileNetV2
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(num_classes, activation="softmax")
])

model_B.summary()

---
## Step 6B — Compile and Train (Fine-Tuning)

**Critical difference:** we use a **much smaller learning rate** — `1e-4` (0.0001) instead of `1e-3` (0.001).

Why? The unfrozen pretrained layers already have good weights from ImageNet training. If we use a large learning rate, the gradient updates would be too large and would overwrite these good weights — completely defeating the purpose of transfer learning. A small learning rate ensures we only make small, careful adjustments to adapt these layers to food images.

We train for **10 epochs** — but because more parameters are updating, each epoch takes slightly longer than Method A.

In [ ]:
model_B.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # VERY SMALL — critical for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Training Method B — Fine-Tuning (last 30 layers unfrozen)...\n")
history_B = model_B.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

---
## Step 7B — Plot Training Curves (Fine-Tuning)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_B.history["accuracy"],     label="Train")
plt.plot(history_B.history["val_accuracy"], label="Validation", linestyle="--")
plt.title("Method B — Accuracy (Fine-Tuning)")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_B.history["loss"],     label="Train")
plt.plot(history_B.history["val_loss"], label="Validation", linestyle="--")
plt.title("Method B — Loss (Fine-Tuning)")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 8B — Evaluate and Confusion Matrix (Fine-Tuning)

In [ ]:
test_loss_B, test_acc_B = model_B.evaluate(test_generator)
print(f"[Method B] Test Loss:     {test_loss_B:.4f}")
print(f"[Method B] Test Accuracy: {test_acc_B:.4f} ({test_acc_B*100:.1f}%)")

test_generator.reset()
preds_B = model_B.predict(test_generator)
y_pred_B = np.argmax(preds_B, axis=1)

cm_B = confusion_matrix(y_true, y_pred_B)
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay(cm_B, display_labels=labels).plot(cmap="Greens", xticks_rotation=45)
plt.title("Confusion Matrix — Method B (Fine-Tuning)")
plt.tight_layout()
plt.show()

print("\nClassification Report — Method B:")
print(classification_report(y_true, y_pred_B, target_names=labels))


---

## Questions

1. **Why does Method A (frozen) converge so much faster than training a CNN from scratch?**

2. **In Method A, what would happen if you used `rescale=1./255` instead of `preprocess_input`? Why is this a problem?**

3. **Why must the learning rate be very small in Method B (fine-tuning)? What happens if you set it to 0.01?** *(Run Exp B3 to find out)*

4. **Compare the confusion matrices of Method A and Method B. Which class improved the most? Why might fine-tuning help for that class?**

5. **Which method would you choose if you had only 50 images per class? What about 5000 images per class?**

6. **What is `GlobalAveragePooling2D` and why is it preferred over `Flatten` in transfer learning architectures?**

---
*TP 3 — Transfer Learning 